In [1]:
!pip install biopython

In [2]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

In [3]:
import requests
import re
import subprocess
from Bio import SeqIO


class MyFastaParser:
    def __init__(self, fasta_file):
        self.fasta_file = fasta_file
        self._warning = set()

    #  SEQKIT
    def seqkit_stats(self):
        try:
            result = subprocess.run(
                ['seqkit', 'stats', '-a', '-T', self.fasta_file],
                capture_output=True,
                text=True,
                check=True
            )

            lines = result.stdout.strip().split('\n')

            headers = lines[0].split('\t')
            values = lines[1].split('\t')

            stat_info = dict(zip(headers, values))

            return {
                'fasta_seqkit_stat_info': stat_info,
                'fasta_type': stat_info.get('type', 'Unknown'),
                'fasta_num_seqs': int(stat_info.get('num_seqs', 0))
            }

        except Exception as e:
            return {'error': str(e)}

    #  API
    def _get_uniprot(self, accession):
        try:
            url = f"https://rest.uniprot.org/uniprotkb/{accession}"
            r = requests.get(url, headers={"Accept": "application/json"})
            return r.json() if r.status_code == 200 else {"error": r.status_code}
        except Exception as e:
            return {"error": str(e)}

    def _get_ensembl(self, ensembl_id):
        try:
            clean_id = ensembl_id.split('.')[0]
            url = f"https://rest.ensembl.org/lookup/id/{clean_id}"
            r = requests.get(url, headers={"Content-Type": "application/json"})
            return r.json() if r.status_code == 200 else {"error": r.status_code}
        except Exception as e:
            return {"error": str(e)}

    #  PARSING
    def _parse_uniprot(self, data):
        if "error" in data:
            return data

        try:
            organism = data.get('organism', {}).get('scientificName', 'unknown')

            gene_info = []
            for gene in data.get('genes', []):
                entry = {}

                if 'geneName' in gene:
                    entry['geneName'] = {
                        'value': gene['geneName'].get('value', 'unknown')
                    }

                if 'synonyms' in gene:
                    entry['synonyms'] = [
                        {'value': s.get('value', 'unknown')}
                        for s in gene['synonyms']
                    ]

                if entry:
                    gene_info.append(entry)

            if not gene_info:
                gene_info = [{'geneName': {'value': 'unknown'}}]

            seq = data.get('sequence', {})

            return {
                'organism': organism,
                'geneInfo': gene_info,
                'sequenceInfo': {
                    'value': seq.get('value', ''),
                    'length': seq.get('length', 0),
                    'molWeight': seq.get('molWeight', 0),
                    'crc64': seq.get('crc64', ''),
                    'md5': seq.get('md5', '')
                },
                'type': 'protein'
            }

        except Exception as e:
            return {"error": str(e)}

    def _parse_ensembl(self, data, sequence):
        if "error" in data:
            return data

        try:
            return {
                'organism': data.get('species', 'unknown'),
                'geneInfo': [{
                    'geneName': {'value': data.get('display_name', 'unknown')}
                }],
                'sequenceInfo': {
                    'value': sequence,
                    'length': len(sequence),
                    'molWeight': len(sequence) * 650,
                    'crc64': '',
                    'md5': ''
                },
                'type': 'dna'
            }
        except Exception as e:
            return {"error": str(e)}

    #  REGEX
    def _extract_uniprot(self, desc):
        match = re.search(r'(?:sp|tr)\|([A-Z0-9]+)\|', desc)
        return match.group(1) if match else None

    def _extract_ensembl(self, desc):
        match = re.search(r'(ENS[A-Z]*\d+(\.\d+)?)', desc)
        return match.group(1) if match else None

    # MAIN PARSER
    def biopython_parser(self, stats):
        if 'error' in stats:
            return stats

        fasta_type = stats['fasta_type']

        if fasta_type == "Protein":
            db = "uniprot"
        elif fasta_type in ["DNA", "RNA"]:
            db = "ensembl"
        else:
            db = "unknown"

        output = {"DB_name": db}

        for record in SeqIO.parse(self.fasta_file, "fasta"):
            desc = record.description
            seq = str(record.seq)

            if db == "uniprot":
                id_ = self._extract_uniprot(desc)
            else:
                id_ = self._extract_ensembl(desc)

            if not id_:
                self._warning.add("No ID match found.")
                continue

            # file_info
            output[f"file_info_{id_}"] = {
                "description": desc,
                "sequence": seq
            }

            # database_info
            if db == "uniprot":
                data = self._parse_uniprot(self._get_uniprot(id_))
            else:
                data = self._parse_ensembl(self._get_ensembl(id_), seq)

            output[f"database_info_{id_}"] = data

        return output

    # OUTPUT
    def show_output(self, data):
        print("> DB_name")
        print(f"\t{data['DB_name']}")

        for k, v in data.items():
            if k == "DB_name":
                continue

            print(k)

            if isinstance(v, dict):
                for k2, v2 in v.items():
                    print(f"\t{k2}")
                    if isinstance(v2, dict):
                        for k3, v3 in v2.items():
                            print(f"\t\t{k3}")
                            print(f"\t\t\t{v3}")
                    else:
                        print(f"\t\t{v2}")

        if self._warning:
            print("WARNING")
            print(f"\t{self._warning}")

In [4]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(json.dumps(stats, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "test_file.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "2",
    "sum_len": "456",
    "min_len": "29",
    "avg_len": "228.0",
    "max_len": "427",
    "Q1": "29",
    "Q2": "228",
    "Q3": "427",
    "sum_gap": "0",
    "N50": "427",
    "N50_num": "1",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 2
}


In [5]:
biopython = parser.biopython_parser(stats)

parser.show_output(biopython)

> DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNES